# 0. Node Mapping

This notebook creates LMDB databases to map raw node IDs (strings/ints) to contiguous integer IDs (0 to N-1).
These mappings are crucial for efficient graph processing and are used in subsequent indexing steps.

In [3]:
# Configuration
import os

# EXTERNAL DRIVE CONFIGURATION
ROOT_DIR = "/Volumes/Backup Plus/Zaman/graph"
DATA_DIR = os.path.join(ROOT_DIR, "data")
OUTPUT_LMDB_DIR = os.path.join(ROOT_DIR, "lmdb_node_mapping")

# Dictionary mapping Node Type -> Parquet Directory
NODE_TABLES = {
    "nasabah": os.path.join(DATA_DIR, "node_nasabah"),
    "pekerja": os.path.join(DATA_DIR, "node_pekerja"),
    "pinjaman": os.path.join(DATA_DIR, "node_pinjaman"),
    "simpanan": os.path.join(DATA_DIR, "node_simpanan"),
    "transaksi": os.path.join(DATA_DIR, "node_transaksi"),
}

# Dictionary mapping Node Type -> Unique ID Column Name in Parquet
ID_COLUMN = {
    "nasabah": "cif",
    "pekerja": "pn",
    "pinjaman": "acctno",
    "simpanan": "acctno",
    "transaksi": "id_trx"
}

# Estimated Map Size (Bytes) for LMDB
# Adjust these if you get MapFullError
MAP_SIZE = {
    "nasabah": 1024 * 1024 * 1024 * 25,  # ~25GB
    "pekerja": 1024 * 1024 * 1024 * 1,   # ~1GB
    "pinjaman": 1024 * 1024 * 1024 * 5,  # ~5GB
    "simpanan": 1024 * 1024 * 1024 * 40, # ~40GB
    "transaksi": 1024 * 1024 * 1024 * 10 # ~10GB
}

BATCH_SIZE = 75_000

os.makedirs(OUTPUT_LMDB_DIR, exist_ok=True)

In [4]:
# Imports
import lmdb
import pyarrow as pa
import pyarrow.dataset as ds
from tqdm.notebook import tqdm
import json
import shutil

In [5]:
# Utilities
def encode(x: str) -> bytes:
    return str(x).encode()

# NOTE: Removed patch_schema function as it was causing issues with PyArrow reading.
# Python's str() Handles Decimal conversion implicitly during text processing loop.

In [6]:
def build_node_mapping():
    counters = {}
    
    for node_type, folder in NODE_TABLES.items():
        if not os.path.exists(folder):
            print(f"Warning: Skipping {node_type} - Folder not found: {folder}")
            continue
            
        print(f"\n=== Processing Node: {node_type} ===")
        
        lmdb_path = os.path.join(OUTPUT_LMDB_DIR, f"{node_type}.lmdb")
        
        # Remove existing LMDB to ensure clean build - ROBUST METHOD
        if os.path.exists(lmdb_path):
            try:
                # ignore_errors=True prevents crash if file is missing/locked during delete
                shutil.rmtree(lmdb_path, ignore_errors=True)
                print(f"  Removed existing database: {lmdb_path}")
            except NotADirectoryError:
                try:
                    os.remove(lmdb_path)
                    print(f"  Removed existing file: {lmdb_path}")
                except FileNotFoundError:
                    pass

        # Open LMDB
        env = lmdb.open(
            lmdb_path,
            map_size=MAP_SIZE.get(node_type, 1024**3), # Default 1GB if not specified
            subdir=True,
            lock=True,
            readonly=False,
            max_dbs=1,
        )

        try:
            # Simplified loading: Auto-detect partitioning and schema
            dataset = ds.dataset(folder, format="parquet")
        except Exception as e:
            print(f"Error opening dataset for {node_type}: {e}")
            env.close()
            continue

        id_col = ID_COLUMN[node_type]
        counter = 0
        
        # Check if ID column exists
        if id_col not in dataset.schema.names:
             print(f"Error: ID column '{id_col}' not found in {node_type} schema: {dataset.schema.names}")
             env.close()
             continue

        # Write to LMDB
        txn = env.begin(write=True)
        
        # Iterate over batches
        pbar = tqdm(dataset.to_batches(columns=[id_col], batch_size=BATCH_SIZE), desc=node_type)
        total_processed = 0
        first_batch_debug = True
        
        for batch in pbar:
            ids = batch[0].to_pylist()
            total_processed += len(ids)
            
            # DEBUG: Print first few IDs to verify data being read
            if first_batch_debug and len(ids) > 0:
                print(f"[DEBUG] First 5 IDs: {ids[:5]}")
                first_batch_debug = False

            for node_id in ids:
                if node_id is None:
                    continue
                
                key = encode(node_id)
                
                # Only insert if key doesn't exist (handle duplicates if any)
                if txn.get(key) is None:
                    txn.put(key, encode(counter))
                    counter += 1
            
            # Periodically commit and update display
            if counter % 500_000 == 0:
                txn.commit()
                txn = env.begin(write=True)
                pbar.set_postfix({"Mapped": f"{counter:,}", "Total": f"{total_processed:,}"})
        
        # Final commit
        txn.commit()
        env.close()
        
        counters[node_type] = counter
        print(f"Total unique mapped nodes for {node_type}: {counter:,}")
        print(f"Saved to {lmdb_path}")

    return counters

In [7]:
if __name__ == "__main__":
    node_counts = build_node_mapping()
    print("\nSummary:")
    print(json.dumps(node_counts, indent=4))


=== Processing Node: nasabah ===
  Removed existing database: /Volumes/Backup Plus/Zaman/graph/lmdb_node_mapping/nasabah.lmdb


nasabah: 0it [00:00, ?it/s]

[DEBUG] First 5 IDs: ['A370298', 'A370298', 'A378924', 'A378924', 'A379437']
Total unique mapped nodes for nasabah: 12,270,075
Saved to /Volumes/Backup Plus/Zaman/graph/lmdb_node_mapping/nasabah.lmdb

=== Processing Node: pekerja ===
  Removed existing database: /Volumes/Backup Plus/Zaman/graph/lmdb_node_mapping/pekerja.lmdb


pekerja: 0it [00:00, ?it/s]

[DEBUG] First 5 IDs: [295785, 154939, 58341, 237727, 54906]
Total unique mapped nodes for pekerja: 6,250
Saved to /Volumes/Backup Plus/Zaman/graph/lmdb_node_mapping/pekerja.lmdb

=== Processing Node: pinjaman ===
  Removed existing database: /Volumes/Backup Plus/Zaman/graph/lmdb_node_mapping/pinjaman.lmdb


pinjaman: 0it [00:00, ?it/s]

[DEBUG] First 5 IDs: [Decimal('779101004900107'), Decimal('779101003007100'), Decimal('779101004228107'), Decimal('779101002606105'), Decimal('779101004835108')]
Total unique mapped nodes for pinjaman: 1,524,589
Saved to /Volumes/Backup Plus/Zaman/graph/lmdb_node_mapping/pinjaman.lmdb

=== Processing Node: simpanan ===
  Removed existing database: /Volumes/Backup Plus/Zaman/graph/lmdb_node_mapping/simpanan.lmdb


simpanan: 0it [00:00, ?it/s]

[DEBUG] First 5 IDs: [Decimal('779101000856534'), Decimal('779101001884522'), Decimal('779101009734535'), Decimal('779101005557509'), Decimal('779101007019523')]


KeyboardInterrupt: 